In [ ]:
import numpy as np
from scipy.integrate import fixed_quad
import plotly.graph_objects as go
import pandas as pd
from iminuit import Minuit
from iminuit.cost import LeastSquares
import matplotlib.pyplot as plt
import os 
import time 

In [ ]:
save_folder = 'run7'
n_points = 10000

lower_factor = 0.99
upper_factor = 2 - lower_factor

In [ ]:
# Load experimental data
atlas_data = pd.read_csv('../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)
x_totem, y_totem, yerr_totem = process_data(totem_data, totem_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

x_7_totem, y_7_totem, yerr_7_totem = x_totem[0], y_totem[0], yerr_totem[0]
x_8_totem, y_8_totem, yerr_8_totem = x_totem[1], y_totem[1], yerr_totem[1]
x_13_totem, y_13_totem, yerr_13_totem = x_totem[2], y_totem[2], yerr_totem[2]

In [ ]:

b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689,
            'a2': 1.7
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'   

def get_parameters_with_variations(ensemble_parameters, ensemble_name, model_type, lower_factor=lower_factor, upper_factor=upper_factor):
    # Obtém os parâmetros iniciais
    initial_params = ensemble_parameters[ensemble_name][model_type]
    
    # Cria as variações
    initial_params_low = {k: v * lower_factor for k, v in initial_params.items()}
    initial_params_high = {k: v * upper_factor for k, v in initial_params.items()}
    
    return initial_params, initial_params_low, initial_params_high

# Get parameters for selected configuration
initial_params_log_atlas = ensemble_parameters[ensemble_atlas][log_model_type]
initial_params_pl_atlas = ensemble_parameters[ensemble_atlas][pl_model_type]

# Para Atlas
initial_params_log_atlas, initial_params_low_log_atlas, initial_params_high_log_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, log_model_type)

initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, pl_model_type)

# Para Totem
initial_params_log_totem, initial_params_low_log_totem, initial_params_high_log_totem = \
    get_parameters_with_variations(ensemble_parameters, ensemble_totem, log_model_type)

initial_params_pl_totem, initial_params_low_pl_totem, initial_params_high_pl_totem = \
    get_parameters_with_variations(ensemble_parameters, ensemble_totem, pl_model_type)



In [ ]:
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  

def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323


In [ ]:
def model_function(x, eps, mg, a1, a2, sqrt_s, model_type='log'):

    # Definindo os parâmetros específicos do modelo
    params = {
        'epsilon': eps,
        'mg': mg,
        'a1': a1,
        'a2': a2
    }
    
    # Escolhendo a massa conforme o modelo
    m2 = m2_log if model_type == 'log' else m2_pl
    
    dif_sigma_lst = []
    
    for q2 in x:
        t = -q2
        
        def inner_integral(x_inner):
            return fixed_quad(
                lambda y: integrand(y, x_inner, params['mg'], params['a1'], 
                                  params['a2'], m2, q2, sqrt_s),
                0, 1,
                n=n_points
            )[0]

        integral_value = fixed_quad(
            inner_integral,
            0, 1,
            n=n_points
        )[0]

        diff_T = integral_value
        s = sqrt_s ** 2
        amp_value = amp_calculation(diff_T, s, params['epsilon'], t)
        dif_sigma_value = differential_sigma(amp_value, s)
        dif_sigma_lst.append(dif_sigma_value)
    
    return np.array(dif_sigma_lst)

In [ ]:
def create_least_squares(x, y, yerr, sqrt_s, model_type):
    return LeastSquares(x, y, yerr, 
        lambda x, eps, mg, a1, a2: model_function(x, eps, mg, a1, a2, sqrt_s, model_type))

def total_cost(dataset, model_type):
    return sum([
        create_least_squares(*dataset[7000], 7000, model_type),
        create_least_squares(*dataset[8000], 8000, model_type),
        create_least_squares(*dataset[13000], 13000, model_type)
    ])

# Estrutura dos dados por experimento
atlas_data = {
    7000: (x_7_atlas, y_7_atlas, yerr_7_atlas),
    8000: (x_8_atlas, y_8_atlas, yerr_8_atlas),
    13000: (x_13_atlas, y_13_atlas, yerr_13_atlas)
}

totem_data = {
    7000: (x_7_totem, y_7_totem, yerr_7_totem),
    8000: (x_8_totem, y_8_totem, yerr_8_totem),
    13000: (x_13_totem, y_13_totem, yerr_13_totem)
}

# Custos totais
total_cost_log_atlas = total_cost(atlas_data, 'log')
total_cost_pl_atlas = total_cost(atlas_data, 'pl')
total_cost_log_totem = total_cost(totem_data, 'log')
total_cost_pl_totem = total_cost(totem_data, 'pl')


In [ ]:
import time
from iminuit import Minuit

def otimization(total_cost_func, initial_params, initial_params_low, initial_params_high, 
                model_type: str, ensemble: str, output_dir: str):
    print('\n')
    print(80 * '-')
    print(f"Iniciando otimização dos parâmetros para {model_type} em {ensemble.upper()}")

    # Todas combinações de parâmetros fixados (1) ou não (0) como dicionários explícitos
    limit_combinations = [
        {'mg': 0, 'eps': 0, 'a1': 0, 'a2': 0},
        {'mg': 0, 'eps': 0, 'a1': 0, 'a2': 1},
        {'mg': 0, 'eps': 0, 'a1': 1, 'a2': 0},
        {'mg': 0, 'eps': 0, 'a1': 1, 'a2': 1},
        {'mg': 0, 'eps': 1, 'a1': 0, 'a2': 0},
        {'mg': 0, 'eps': 1, 'a1': 0, 'a2': 1},
        {'mg': 0, 'eps': 1, 'a1': 1, 'a2': 0},
        {'mg': 0, 'eps': 1, 'a1': 1, 'a2': 1},
        {'mg': 1, 'eps': 0, 'a1': 0, 'a2': 0},
        {'mg': 1, 'eps': 0, 'a1': 0, 'a2': 1},
        {'mg': 1, 'eps': 0, 'a1': 1, 'a2': 0},
        {'mg': 1, 'eps': 0, 'a1': 1, 'a2': 1},
        {'mg': 1, 'eps': 1, 'a1': 0, 'a2': 0},
        {'mg': 1, 'eps': 1, 'a1': 0, 'a2': 1},
        {'mg': 1, 'eps': 1, 'a1': 1, 'a2': 0},
        {'mg': 1, 'eps': 1, 'a1': 1, 'a2': 1},
    ]

    # Configurações para cada modelo
    if model_type == 'log':
        param_limits = {
            'mg': (0.3, 0.45),
            'eps': (0.06, 0.09),
            'a1': (1.2, 1.7),
            'a2': (1.5, 3.5)
        }
    elif model_type == 'pl':
        param_limits = {
            'mg': (0.35, 0.5),
            'eps': (0.06, 0.09),
            'a1': (1.2, 2),
            'a2': (1.5, 3)
        }
    else:
        raise ValueError("model_type deve ser 'log' ou 'pl'")

    # Valores para varrer
    down_values = [round(0.9 + i*0.01, 2) for i in range(10)]  # 0.9 a 0.99
    strategies = [0, 1, 2]
    ncalls = [200, 300, 400, 500, 600, 700, 800]

    for combo in limit_combinations:
        # Nome do arquivo de saída para esta combinação de limites
        filename = f"resultados_otimizacao_{model_type}_{ensemble.lower()}_eps_{combo['eps']}_mg_{combo['mg']}_a1_{combo['a1']}_a2_{combo['a2']}.txt"
        filepath = f"{output_dir}/{filename}"
        
        # Abre o arquivo uma vez para esta combinação de limites
        with open(filepath, 'w') as f:
            # Escreve o cabeçalho
            f.write(f"Otimização para {model_type} em {ensemble.upper()} - Limites: eps_{combo['eps']}_mg_{combo['mg']}_a1_{combo['a1']}_a2_{combo['a2']}\n")
            f.write("Configuração: down | up | strategy | migrad | mg ± error | eps ± error | a1 ± error | a2 ± error | χ²/ndof\n")
            f.write("="*120 + "\n")
            
            for down in down_values:
                up = 2 - down
                for strategy in strategies:
                    for ncall in ncalls:
                        start_time = time.time()
                        
                        print(f"\nExecutando combinação: mg_{combo['mg']}_eps_{combo['eps']}_"
                              f"a1_{combo['a1']}_a2_{combo['a2']}")
                        print(f"down: {down}, up: {up}, strategy: {strategy}, ncall: {ncall}")

                        # Criar instância do Minuit
                        m = Minuit(total_cost_func,
                                  eps=initial_params['epsilon'],
                                  mg=initial_params['mg'],
                                  a1=initial_params['a1'],
                                  a2=initial_params['a2'])

                        # Configurações comuns
                        m.errordef = 7.79
                        m.strategy = strategy
                        m.migrad(ncall=ncall)
                        m.hesse()
                        
                        # Aplicar fixação e limites
                        for param in ['mg', 'eps', 'a1', 'a2']:
                            if combo[param] == 1:
                                m.fixed[param] = True
                            else:
                                m.limits[param] = param_limits[param]

                        # Otimização em duas etapas
                        try:
                            # Primeiro ajuste: apenas parâmetros não fixados
                            m.migrad(ncall=ncall//2)
                            
                            # Segunda etapa: tentar ajustar todos (mesmo os fixos)
                            m.fixed = False  # Libera todos parâmetros
                            m.migrad(ncall=ncall//2)
                            m.hesse()
                            
                            execution_time = time.time() - start_time
                            
                            if m.valid:
                                chi2_ndof = m.fval / m.ndof

                                if 0.3 < chi2_ndof < 1.2:
                                    # Escrever resultados
                                    line = (
                                        f"{down:.2f} | {up:.2f} | {strategy} | {ncall} | "
                                        f"{m.values['mg']:.6f} ± {m.errors['mg']:.6f} | "
                                        f"{m.values['eps']:.6f} ± {m.errors['eps']:.6f} | "
                                        f"{m.values['a1']:.6f} ± {m.errors['a1']:.6f} | "
                                        f"{m.values['a2']:.6f} ± {m.errors['a2']:.6f} | "
                                        f"{chi2_ndof:.6f}\n"
                                    )
                                    f.write(line)
                                    f.flush()
                                    
                                    print(f"Convergido em {execution_time:.2f}s - χ²/ndof: {chi2_ndof:.4f}")
                            else:
                                print("Minuit não convergiu")
                                
                        except Exception as e:
                            print(f"Erro durante otimização: {str(e)}")
                            continue

    print("\nOtimização concluída para todas as combinações!")

In [ ]:
model = 'pl'
ensemble = 'atlas'

# Defina o diretório de saída (deve existir)
output_directory = f"../../results/all_possible_iterations/all_possible_iterations_{model}_{ensemble}_v5"
os.makedirs(output_directory, exist_ok=True)

# Execute a otimização
otimization(total_cost_pl_atlas, initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas,
           model_type=model, ensemble=ensemble, output_dir=output_directory)